## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [42]:
from dotenv import load_dotenv
from langchain_openai import OpenAI
from langchain_ollama import ChatOllama
#from langchain_community.chat_models import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
from LLMClient import get_client, get_models
import gradio as gr

In [58]:
MODELS = get_models("ollama")
MODEL = MODELS["flagship"]
DB_NAME = "vector_db"
load_dotenv(override=True)

MODEL

'nemotron-3-ultra'

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [53]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [60]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(
    model=MODEL,
    temperature=0,
    base_url="https://ollama.com",
)

### These LangChain objects implement the method `invoke()`

In [61]:
retriever.invoke("Who is Avery?")

[Document(id='f746e8cf-bcd1-43e1-8adb-0e35d83402de', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [62]:
llm.invoke("Who is Avery?")

AIMessage(content='"Avery" is a common name that can refer to many different people, characters, places, or things. Without more context, here are the most notable possibilities:\n\n### **Real People (Historical & Contemporary)**\n*   **James Avery (1945–2013):** Beloved American actor best known as **Uncle Phil** on *The Fresh Prince of Bel-Air* and the voice of Shredder in the original *Teenage Mutant Ninja Turtles* cartoon.\n*   **Jackson Avery:** A fictional character (see below), but often confused with the actor **Jesse Williams** who plays him.\n*   **Montgomery Avery / Avery Brooks:** Actor best known as **Captain Benjamin Sisko** on *Star Trek: Deep Space Nine*.\n*   **Sean Avery:** Former NHL hockey player known for his agitator style and fashion ventures.\n*   **John Avery (pirate):** A notorious 17th-century English pirate (also known as Henry Every).\n\n### **Fictional Characters**\n*   **Dr. Jackson Avery:** A major character on *Grey\'s Anatomy* (played by Jesse Williams

## Time to put this together!

In [63]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [64]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [65]:
answer_question("Who is Averi Lancaster?", [])

'Based on the information provided, **Avery Lancaster** (spelled with a "y") is the **Co-Founder & Chief Executive Officer (CEO)** of Insurellm.\n\nHere are the key details from her profile:\n\n*   **Role:** Co-Founder & CEO (2015 – Present)\n*   **Location:** San Francisco, California\n*   **Current Salary:** $225,000\n*   **Background:** Before founding Insurellm, she served as a **Senior Product Manager at Innovate Insurance Solutions** (2013–2015), where she developed groundbreaking insurance products for the tech sector.\n*   **Reputation:** She is known for her innovative leadership strategies and risk management expertise, which have established Insurellm as a leading Insurance Tech provider.\n\nThere is no "Averi Lancaster" (with an "i") listed in the provided context.'

## What could possibly come next? 😂

In [66]:
gr.ChatInterface(answer_question).launch()

/Users/dpulache/dev/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!